## Import modules and libraries

In [ ]:
from pathlib import Path
import pycolmap
import random
import pickle

from hloc import visualization
from hloc.utils import viz_3d

## Load paths

In [ ]:
running_set = 'megaloc_superpoint_lightglue'
# running_set = 'megaloc_disk_lightglue'
# running_set = 'megaloc_aliked_lightglue'
# running_set = 'megaloc_loftr'
# running_set = 'megaloc_loma'
query_set = 'query1'

PIXLOC_ROOT = Path.cwd().parent

DB_IMG_DIR = PIXLOC_ROOT / Path("datasets/zedx_mini/images/db")
QUERY_IMG_DIR = PIXLOC_ROOT / Path("datasets/zedx_mini/images") / query_set

LOCALIZATION_DIR = PIXLOC_ROOT / Path("outputs/hloc/zedx_mini") / running_set
MODEL_DIR = LOCALIZATION_DIR / "sfm_model"
QUERY_LOC_FILE = LOCALIZATION_DIR / query_set / "query_loc.txt"

## Load the SfM model

In [ ]:
reconstruction = pycolmap.Reconstruction(str(MODEL_DIR))
print("Cameras:", len(reconstruction.cameras))
print("Images:", len(reconstruction.images))
print("Points3D:", len(reconstruction.points3D))

## Load the logs

In [ ]:
with open(str(QUERY_LOC_FILE) + "_logs.pkl", "rb") as f:
    logs = pickle.load(f)

if isinstance(logs, dict):
    print("Top-level keys:", list(logs.keys()))
if "loc" in logs:
    print("Num queries:", len(logs["loc"]))
    qname = next(iter(logs["loc"]))
    loc = logs["loc"][qname]
    if isinstance(loc, dict):
        print("Fields:")
        for k, v in loc.items():
            msg = f"  {k}: {type(v).__name__}"
            if hasattr(v, "shape"):
                msg += f", shape={v.shape}"
            try:
                if hasattr(v, "__len__") and not isinstance(v, str):
                    msg += f", len={len(v)}"
            except Exception:
                pass
            print(msg)

## Visualizing the SfM model
We visualize some of the database images with their detected keypoints.

In [ ]:
seed = random.randint(0, 1000000)

# Color the keypoints by track length: red keypoints are observed many times, blue keypoints few
visualization.visualize_sfm_2d(reconstruction, DB_IMG_DIR, n=1, seed=seed, color_by="track_length")

# Color the keypoints by visibility: blue if sucessfully triangulated, red if never matched
visualization.visualize_sfm_2d(reconstruction, DB_IMG_DIR, n=1, seed=seed, color_by="visibility")

# Color the keypoints by triangulated depth: red keypoints are far away, blue keypoints are closer
visualization.visualize_sfm_2d(reconstruction, DB_IMG_DIR, n=1, seed=seed, color_by="depth")

## Visualize the localization
We parse the localization logs and for each query image plot matches and inliers with a few database images.

First collect all valid and invalid queries:

In [ ]:
VALIDITY_FILE = Path(str(QUERY_LOC_FILE) + "_logs_compact.pkl.txt")
valid_queries = []
invalid_queries = []
with open(VALIDITY_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        qname, valid_str = line.split()
        if valid_str == "True":
            valid_queries.append(qname)
        else:
            invalid_queries.append(qname)
print(f"Valid   : {len(valid_queries)}")
print(f"Invalid : {len(invalid_queries)}")

Then visualize each set separately:

In [ ]:
# Select valid queries
seed = random.randint(0, 1000000)
rng = random.Random(seed)
selected = rng.sample(
    valid_queries,
    min(10, len(valid_queries))
)

In [ ]:
# Select invalid queries
seed = random.randint(0, 1000000)
rng = random.Random(seed)
selected = rng.sample(
    invalid_queries,
    min(10, len(invalid_queries))
)

In [ ]:
# Select specific queries
selected = ['1778982199734781000.png']

In [ ]:
if reconstruction is not None:
    if not isinstance(reconstruction, pycolmap.Reconstruction):
        reconstruction = pycolmap.Reconstruction(reconstruction)
for qname in selected:
    loc = logs["loc"][qname]
    print(qname)
    visualization.visualize_loc_from_log(
        QUERY_IMG_DIR,
        qname,
        loc,
        reconstruction,
        DB_IMG_DIR,
        top_k_db=1,
    )